# Fórmulas de Agrometeorologia em Python

**Fonte:** Pereira, Angelocci & Sentelhas (2002) — *Agrometeorologia: fundamentos e aplicações práticas*, ESALQ/USP,
complementado por Allen et al. (1998, FAO-56)


In [4]:
import math
from datetime import datetime, date, timedelta
import numpy as np
import pandas as pd
from calendar import monthrange
import matplotlib.pyplot as plt

## Funções auxiliares (trigonometria em graus)

In [5]:
def sind(x):
    """Seno de um ângulo x expresso em GRAUS."""
    return np.sin(np.radians(x))

def cosd(x):
    """Cosseno de um ângulo x expresso em GRAUS."""
    return np.cos(np.radians(x))

def tand(x):
    """Tangente de um ângulo x expresso em GRAUS."""
    return np.tan(np.radians(x))

def acosd(x):
    """Arco-cosseno que retorna o ângulo em GRAUS (em vez de radianos)."""
    return np.degrees(np.arccos(x))

# Capítulo 1 — Funções

## 1.5 — Capítulo 5: Radiação Solar

In [6]:
# Número do Dia do Ano - NDA
def nda(dia, mes, ano=2023):
    """
    Retorna o Número do Dia do Ano (NDA).

    Parâmetros
    ----------
    dia : int
        Dia do mês (1–31).
    mes : int
        Mês (1–12).
    ano : int, opcional
        Ano. Padrão é 2023 (não bissexto).

    Retorna
    -------
    int
        Número do dia no ano (1 a 365).

    Exemplos
    --------
    >>> nda(1, 1)
    1
    >>> nda(25, 12)
    359
    >>> nda(29, 2, 2024)
    60
    """
    dt = datetime(ano, mes, dia)
    return dt.timetuple().tm_yday

In [7]:
#Declinação solar - d
def declinacao_solar(NDA):
    """
    Declinação solar (delta) para um dado dia do ano.

    Aproximação senoidal da variação anual da declinação solar, decorrente
    da inclinação do eixo de rotação da Terra (23,45°). Positiva quando o
    Sol está aparentemente no hemisfério norte, negativa no hemisfério sul.

    Parâmetros
    ----------
    NDA : int
        Número do dia do ano (1/jan = 1, 1/fev = 32, ..., 31/dez = 365).

    Retorna
    -------
    delta : float
        Declinação solar, em graus.
    """
    return 23.45 * sind(360 * (NDA - 80) / 365)


In [8]:
# Ângulo horário - h
def angulo_horario(hora, minuto=0):
    """
    Converte a hora local (hora solar verdadeira) em ângulo horário.

    A Terra gira 360° em 24h, ou seja, 15° por hora. O ângulo horário é nulo
    (h = 0°) exatamente ao meio-dia local, quando o Sol cruza o meridiano do
    observador; é negativo pela manhã e positivo à tarde.

    Parâmetros
    ----------
    hora : int
        Hora do dia (0–23).
    minuto : int, opcional
        Minutos (0–59). Padrão é 0.

    Retorna
    -------
    h : float
        Ângulo horário, em graus.
    """
    hora_decimal = hora + minuto / 60
    return (hora_decimal - 12) * 15


In [9]:
# Ângulo zenital - Z
def angulo_zenital(lat, declinacao):
    """
    Ângulo zenital do Sol.

    Obtida da equação geral do ângulo zenital (eq. 5.4) fazendo-se h = 0°
    (meio-dia local). Representa a menor altura zenital (posição mais alta
    do Sol) atingida no dia.

    Parâmetros
    ----------
    lat : float
        Latitude do local, em graus (negativa no hemisfério sul).
    declinacao : float
        Declinação solar do dia, em graus.

    Retorna
    -------
    Z : float
        Ângulo zenital, em graus.
    """
    cos_Z = sind(lat) * sind(declinacao) + cosd(lat) * cosd(declinacao)
    return acosd(cos_Z)

def angulo_zenital(lat, declinacao):
    cos_Z = sind(lat) * sind(declinacao) + cosd(lat) * cosd(declinacao)
    cos_Z = max(-1.0, min(1.0, cos_Z))  # evita erro de domínio no acosd
    return acosd(cos_Z)


In [10]:
# Azimute solar - alfa
def azimute_solar(lat, declinacao, Z):
    """
    Azimute solar (alpha): direção horizontal do Sol em
    relação à linha Norte-Sul (referência Sul, no hemisfério sul).

    Parâmetros
    ----------
    lat : float
        Latitude do local, em graus.
    declinacao : float
        Declinação solar do dia, em graus.
    Z : float
        Ângulo zenital (eq. 5.4), em graus.

    Retorna
    -------
    alpha : float
        Azimute solar, em graus.
    """
    num = sind(lat) * cosd(Z) - sind(declinacao)
    den = cosd(lat) * sind(Z)
    return acosd(num / den)


In [11]:
# Comprimento da sombra - S
def comprimento_sombra(d, Z):
    """
    Comprimento da sombra (S) projetada por um objeto de altura d.

    Quanto maior o ângulo zenital (Sol mais baixo no horizonte), maior a
    tangente de Z e, portanto, mais longa a sombra.

    Parâmetros
    ----------
    d : float
        Altura do objeto (m).
    Z : float
        Ângulo zenital no instante considerado, em graus.

    Retorna
    -------
    S : float
        Comprimento da sombra, na mesma unidade de d.
    """
    return d * tand(Z)


In [12]:
# Fotoperíodo - N
def fotoperiodo(Hn):
    """
    Fotoperíodo (N), ou duração do dia, a partir do ângulo
    horário do nascer do Sol.

    Decorre da simetria da trajetória solar em relação ao meio-dia: o
    fotoperíodo é o dobro do ângulo horário do nascer, convertido de graus
    para horas (15°/hora).

    Parâmetros
    ----------
    Hn : float
        Ângulo horário no nascer do Sol, em graus.

    Retorna
    -------
    N : float
        Fotoperíodo, em horas.
    """
    return 2 * Hn / 15


In [13]:
# Ângulo horário no nascer do Sol - Hn
def angulo_horario_nascer(lat, declinacao):
    """
    Ângulo horário no nascer do Sol (Hn).

    Obtida impondo-se Z = 90° (cos Z = 0) na equação geral do ângulo
    zenital, já que no nascer/pôr do Sol o astro está exatamente no
    horizonte.

    Parâmetros
    ----------
    lat : float
        Latitude do local, em graus.
    declinacao : float
        Declinação solar do dia, em graus.

    Retorna
    -------
    Hn : float
        Ângulo horário no nascer do Sol, em graus.
    """
    return acosd(-tand(lat) * tand(declinacao))


## Aplicação 1
____
Calcule o comprimento e a direção da sombra de um poste de 10 m de altura na cidade de Dourados (latitude = 22 S), no dia 2 de março, às 10 horas e 27 minutos. Calcule também o fotoperíodo.


In [14]:
# Dados
d = 10  # Altura do poste, em metros
lat = -22  # Latitude do local, em graus (negativa no hemisfério sul)
dia = 2  # Dia do mês (1-31)
mes = 3  # Mês (1-12)
hora = 10  # Hora do dia (0-23)
minuto = 27  # Minuto do dia (0-59)

#Resolução
NDA = nda(dia, mes)
print(f"Número do Dia do Ano (NDA): {NDA}")
delta = declinacao_solar(NDA)
print(f"Declinação solar (delta): {delta}")
h = angulo_horario(hora, minuto)
print(f"Ângulo horário (h): {h}")
Z = angulo_zenital(lat, delta)
print(f"Ângulo zenital (Z): {Z}")
alfa = azimute_solar(lat, delta, Z)
print(f"Azimute solar (alfa): {alfa}")
S = comprimento_sombra(d, Z)
print(f"Comprimento da sombra (S): {S}")

Hn = angulo_horario_nascer(lat, delta)
print(f"Ângulo horário no nascer do Sol (Hn): {Hn}")
N = fotoperiodo(Hn)
print(f"Fotoperíodo (N): {N}")



Número do Dia do Ano (NDA): 61
Declinação solar (delta): -7.533773566685945
Ângulo horário (h): -23.25000000000001
Ângulo zenital (Z): 14.466226433314066
Azimute solar (alfa): 179.9999973001307
Comprimento da sombra (S): 2.579887953183402
Ângulo horário no nascer do Sol (Hn): 93.06296504278654
Fotoperíodo (N): 12.408395339038206


In [15]:
# Fator de correção -  (d/D)^2
def fator_correcao_distancia(NDA):
    """
    Fator de correção (d/D)^2 da excentricidade da órbita
    terrestre, para o dia do ano considerado.

    Corrige a constante solar em função da variação da distância real
    Terra-Sol (D) em torno da distância média (d = 1 UA) ao longo da
    órbita elíptica terrestre.

    Parâmetros
    ----------
    NDA : int
        Número do dia do ano.

    Retorna
    -------
    (d/D)^2 : float
        Fator de correção adimensional.
    """
    return 1 + 0.033 * cosd(NDA * 360 / 365)


In [16]:
# Radiação solar extraterrestre diária (Qo)
def irradiancia_extraterrestre(lat, declinacao, Hn, dD2):
    """
    Irradiância solar global extraterrestre diária (Qo), no
    topo da atmosfera, para uma superfície horizontal.

    Representa o total diário máximo de energia solar teórica que
    incidiria sobre uma superfície horizontal caso não houvesse atenuação
    atmosférica. Depende apenas da latitude, da declinação solar e da
    correção pela distância Terra-Sol do dia.

    Parâmetros
    ----------
    lat : float
        Latitude do local, em graus.
    declinacao : float
        Declinação solar do dia, em graus.
    Hn : float
        Ângulo horário no nascer do Sol, em graus (eq. 5.20).
    dD2 : float
        Fator de correção (d/D)^2 da distância Terra-Sol (eq. 5.30).

    Retorna
    -------
    Qo : float
        Irradiância solar global extraterrestre diária, em MJ/m² dia.
    """
    hn_rad = np.radians(Hn)
    return 37.6 * dD2 * (hn_rad * sind(lat) * sind(declinacao) + cosd(lat) * cosd(declinacao) * sind(Hn))

In [17]:
# Insolação - insol
def insolacao(N, Tmax, Tmin, lat, k=0.19):
    """
    Estimativa do número de horas de insolação (brilho solar), insol.

    Fórmula empírica baseada na amplitude térmica diária (Tmax - Tmin) e na
    duração astronômica do dia (N), com correção pela latitude local.

    Parâmetros
    ----------
    N : float
        Duração astronômica do dia (fotoperíodo), em horas.
    k : float, opcional
        Coeficiente empírico de ajuste (padrão 0,16 para regiões interioranas;
        0,19 é comumente usado para regiões costeiras).
    Tmax : float
        Temperatura máxima diária, em °C.
    Tmin : float
        Temperatura mínima diária, em °C.
    lat : float
        Latitude do local, em graus (negativa no hemisfério sul).

    Retorna
    -------
    insol : float
        Número de horas de insolação (brilho solar) estimado, em horas.
    """
    insol = (N / 0.52) * (k * (Tmax - Tmin) ** 0.5 - 0.29 * cosd(lat))
    return insol


In [18]:
# Radiação solar global - Qg_AP
def Qg_angstrom(insolacao, N, Qo, lat, b=0.52):
    """
    Equação de Angström-Prescott (variante de Glover-McCulloch): estimativa
    da radiação solar global a partir da razão de insolação n/N.

    Parâmetros
    ----------
    insolacao : float
        Número de horas de brilho solar (insolação) medido no dia, em horas.
    N : float
        Fotoperíodo do dia (número máximo de horas de brilho solar), em horas.
    Qo : float
        Irradiância solar no topo da atmosfera (radiação extraterrestre), em MJ/m².
    lat : float
        Latitude do local, em graus (negativa no hemisfério sul).
    b : float, opcional
        Coeficiente empírico de regressão (padrão 0,52).

    Retorna
    -------
    Qg : float
        Irradiância solar global, em MJ/m².
    """
    a = 0.29 * cosd(lat)
    razao_insolacao = insolacao / N
    Qg = Qo * (a + b * razao_insolacao)
    return Qg

In [19]:
# Radiação solar global (Qg_HS) - Método de Hargreaves-Samani
def Qg_hargreaves(Tmax, Tmin, Qo, k=0.16):
    """
    Equação de Hargreaves-Samani: estimativa da radiação solar global a partir da
    amplitude térmica diária (Tmax - Tmin) e da radiação extraterrestre (Qo).

    Método útil quando não há dados de insolação (n) disponíveis, exigindo
    apenas temperaturas máxima e mínima diárias.

    Parâmetros
    ----------
    Tmax : float
        Temperatura máxima diária, em °C.
    Tmin : float
        Temperatura mínima diária, em °C.
    Qo : float
        Irradiância solar no topo da atmosfera (radiação extraterrestre), em MJ/m².
    k : float, opcional
        Coeficiente empírico de ajuste (padrão 0,16 para regiões interioranas;
        0,19 é comumente usado para regiões costeiras).

    Retorna
    -------
    Qg : float
        Irradiância solar global, em MJ/m².
    """
    Qg = k * (Tmax - Tmin) ** 0.5 * Qo
    return Qg

## Aplicação 2
___
Determine a radiação solar extraterrestre (Qo) e a radiação solar global (Qg) diárias pelo método de Hargreaves-Samani e Angstron-Prescott. Considere a cidade de Foz do Iguaçu-PR (Latitude = -25,60°, atitude = 235,09 m) no dia 21/05.
OBS. Tmax = 21,2 ºC. Tmin = 7,4 ºC; URmax =  97,1% e URmin  = 66,8%


In [20]:
# Dados
Tmax = 21.2
Tmin = 7.4
URmax = 0.971
URmin = 0.668
lat = -25.6
dia = 21
mes = 5
dia = 21
mes = 5

#Resolução
NDA = nda(dia, mes)
print(f"Número do Dia do Ano (NDA): {NDA}")
delta = declinacao_solar(NDA)
print(f"Declinação solar (delta): {delta}")
Hn = angulo_horario_nascer(lat, delta)
print(f"Ângulo horário no nascer do Sol (Hn): {Hn}")
N = fotoperiodo(Hn)
print(f"Fotoperíodo (N): {N}")
dD2 = fator_correcao_distancia(NDA)
print(f"Fator de correção (d/D)^2: {dD2}")
Qo = irradiancia_extraterrestre(lat, delta, Hn, dD2)
print(f"Irradiância solar extraterrestre diária (Qo): {Qo}")
insol = insolacao(N, Tmax, Tmin, lat)
print(f"Número de horas de insolação (brilho solar) estimado (insol): {insol}")
Qg_AP = Qg_angstrom(insol, N, Qo, lat, b=0.52)
print(f"Irradiância solar global (Qg_AP): {Qg_AP}")
Qg_HS = Qg_hargreaves(Tmax, Tmin, Qo)
print(f"Irradiância solar global (Qg_HS): {Qg_HS}")

Número do Dia do Ano (NDA): 141
Declinação solar (delta): 20.341851518409044
Ângulo horário no nascer do Sol (Hn): 79.76827116479618
Fotoperíodo (N): 10.63576948863949
Fator de correção (d/D)^2: 0.9750687206356016
Irradiância solar extraterrestre diária (Qo): 22.84185036361831
Número de horas de insolação (brilho solar) estimado (insol): 9.087185925749393
Irradiância solar global (Qg_AP): 16.122204526178894
Irradiância solar global (Qg_HS): 13.576593285203279


## 1.6 — Capítulo 6: Temperatura

In [21]:
# Temperatura média diária extremos - Tmed
def temp_media_extremos(Tmax, Tmin):
    """
    Temperatura média diária estimada pela média dos valores
    extremos (máxima e mínima).

    Método mais simples e mais usado, porém tende a superestimar a
    temperatura média "real", pois o ritmo diário da temperatura não é
    simétrico em torno da média (fica mais tempo próximo da mínima
    noturna do que da máxima diurna).

    Parâmetros
    ----------
    Tmax, Tmin : float
        Temperaturas máxima e mínima do dia, em °C.

    Retorna
    -------
    Tmed : float
        Temperatura média diária estimada, em °C.
    """
    return (Tmax + Tmin) / 2


In [22]:
# Temperatura média diária automática - Tmed_auto
def temp_media_estacao_automatica(temperaturas):
    """
    Temperatura média a partir de observações de estações
    automáticas (média aritmética simples de N observações no período).

    Quanto maior o número de observações, mais próxima a estimativa fica
    do valor "real" (integral contínua da temperatura ao longo do dia).

    Parâmetros
    ----------
    temperaturas : list[float]
        Lista com as temperaturas (°C) de cada observação (No valores).

    Retorna
    -------
    Tmed : float
        Temperatura média do período, em °C.
    """
    return sum(temperaturas) / len(temperaturas)


## 1.7 — Capítulo 7: Umidade do Ar

In [23]:
# Pressão de saturação - es
def es_tetens(T_ar):
    """
    Equação de Tetens: pressão de saturação de vapor d'água
    (es) em função da temperatura do ar.

    Expressão empírica que fornece a pressão máxima que o vapor d'água
    pode exercer no ar numa dada temperatura, antes de condensar.

    Parâmetros
    ----------
    T_ar : float
        Temperatura do ar, em °C.

    Retorna
    -------
    es : float
        Pressão de saturação de vapor, em kPa.
    """
    return 0.6108 * 10 ** ((7.5 * T_ar) / (237.3 + T_ar))


In [24]:
# Pressão parcial de vapor - ea
def ea_umidade(es, UR):
    """
    Pressão parcial (atual) de vapor d'água (ea) a partir da pressão de
    saturação (es) e da umidade relativa do ar (UR).

    Representa a pressão de vapor d'água realmente exercida na atmosfera,
    proporcional à fração de saturação indicada pela umidade relativa.

    Parâmetros
    ----------
    es : float
        Pressão de saturação de vapor d'água, em kPa.
    UR : float
        Umidade relativa do ar, em porcentagem (ex. 65%).

    Retorna
    -------
    ea : float
        Pressão parcial (atual) de vapor d'água, em kPa.
    """
    return es * (UR/100)

In [25]:
# Déficit de saturação - delta e
def deficit_saturacao(es, ea):
    """
    Déficit de saturação de vapor do ar (delta e).

    Mede o "quanto falta" para o ar atingir a saturação, sendo
    proporcional ao poder evaporante da atmosfera (equivalente ao VPD,
    vapor pressure deficit).

    Parâmetros
    ----------
    es : float
        Pressão de saturação de vapor, em kPa (eq. 7.2).
    ea : float
        Pressão parcial (atual) de vapor d'água, em kPa.

    Retorna
    -------
    delta_e : float
        Déficit de saturação de vapor, em kPa.
    """
    return es - ea


In [26]:
# Pressão atmosférica - Patm
def patm_altitude(A):
    """
    Pressão atmosférica (Patm) em função da altitude do local.

    Equação derivada da lei dos gases ideais, considerando o gradiente
    térmico padrão da atmosfera, para estimar a pressão atmosférica local
    a partir apenas da elevação (altitude) acima do nível do mar.

    Parâmetros
    ----------
    A : float
        Altitude do local, em metros.

    Retorna
    -------
    Patm : float
        Pressão atmosférica local, em kPa.
    """
    return 101.3 * ((293 - 0.0065 * A) / 293) ** 5.26

In [27]:
# Umidade absoluta do ar - UA
def umidade_absoluta(ea, T_ar_C):
    """
    Umidade absoluta do ar (UA): massa de vapor d'água por
    unidade de volume de ar.

    Deriva da equação de estado dos gases ideais aplicada ao vapor
    d'água; a constante 2168 vem da razão entre a massa molar da água e
    a constante universal dos gases.

    Parâmetros
    ----------
    ea : float
        Pressão parcial de vapor d'água, em kPa.
    T_ar_C : float
        Temperatura do ar, em °C (internamente convertida para Kelvin).

    Retorna
    -------
    UA : float
        Umidade absoluta, em g de H2O por m³ de ar.
    """
    T_K = T_ar_C + 273.15
    return 2168 * ea / T_K


In [28]:
# Umidade de saturação do ar - US
def umidade_saturacao(es, T_ar_C):
    """
    Umidade de saturação do ar (US): massa máxima de vapor
    d'água que o ar pode reter por unidade de volume, na temperatura T.

    Calculada do mesmo modo que a umidade absoluta (eq. 7.8), porém
    usando a pressão de saturação (es) no lugar da pressão parcial (ea).

    Parâmetros
    ----------
    es : float
        Pressão de saturação de vapor, em kPa (eq. 7.2).
    T_ar_C : float
        Temperatura do ar, em °C.

    Retorna
    -------
    US : float
        Umidade de saturação, em g de H2O por m³ de ar.
    """
    T_K = T_ar_C + 273.15
    return 2168 * es / T_K


In [29]:
# Umidade relativa do ar - UR%
def umidade_relativa(ea, es):
    """
    Umidade relativa do ar (UR%), razão entre a pressão
    parcial e a pressão de saturação de vapor (equivalente à razão entre
    UA e US).

    Parâmetros
    ----------
    ea : float
        Pressão parcial (atual) de vapor d'água, em kPa.
    es : float
        Pressão de saturação de vapor, em kPa.

    Retorna
    -------
    UR : float
        Umidade relativa do ar, em %.
    """
    return 100 * (ea / es)


In [30]:
# Temperatura do ponto de orvalho - To
def ponto_orvalho(ea):
    """
    Temperatura do ponto de orvalho (To): temperatura à qual
    o ar, mantendo o mesmo teor de vapor d'água, atingiria a saturação.

    Obtida invertendo-se algebricamente a equação de Tetens (eq. 7.2)
    para se obter To a partir de um valor conhecido de ea.

    Parâmetros
    ----------
    ea : float
        Pressão parcial (atual) de vapor d'água, em kPa.

    Retorna
    -------
    To : float
        Temperatura do ponto de orvalho, em °C.
    """
    log_termo = math.log10(ea / 0.6108)
    return (237.3 * log_termo) / (7.5 - log_termo)


In [31]:
# Constante psicrométrica - gamma
def constante_psicrometrica(Patm):
    """
    Constante psicrométrica (gamma).

    Relaciona a pressão parcial de vapor d'água com a temperatura do ar
    numa dada pressão atmosférica, sendo essencial para converter energia
    disponível em déficit de pressão de vapor equivalente.

    Parâmetros
    ----------
    Patm : float
        Pressão atmosférica local, em kPa (eq. 7.6, patm_altitude).

    Retorna
    -------
    gamma : float
        Constante psicrométrica, em kPa/°C.
    """
    return 0.665e-3 * Patm

## 1.8 — Capítulo 8: Balanço de Energia

In [32]:
# Saldo de radiação - Rn
def saldo_radiacao(BOC, BOL):
    """
    Saldo de radiação (Rn), soma do balanço de ondas curtas
    (BOC) com o balanço de ondas longas (BOL).

    Representa a energia radiante líquida disponível numa superfície para
    os processos físicos e biológicos que nela ocorrem.

    Parâmetros
    ----------
    BOC : float
        Balanço de ondas curtas, mesma unidade de Rn.
    BOL : float
        Balanço de ondas longas, mesma unidade de Rn.

    Retorna
    -------
    Rn : float
        Saldo de radiação, em MJ/m² dia (ou W/m²).
    """
    return BOC + BOL


In [33]:
# Balanço de ondas curtas - BOC
def boc_saldo(Qg, r=0.25):
    """
    Balanço de ondas curtas (BOC): irradiância solar global
    incidente menos a parcela refletida pela superfície (albedo).

    Parâmetros
    ----------
    Qg : float
        Irradiância solar global incidente na superfície (MJ/m² dia ou W/m²).
    r : float
        Coeficiente de reflexão da superfície (albedo), adimensional (0-1).
        r = 0.25 média para gramado
    Retorna
    -------
    BOC : float
        Balanço de ondas curtas, mesma unidade de Qg.
    """
    return Qg * (1 - r)


In [34]:
# Balanço de radiação de onda longa - BOL
def bol_saldo(Tmax, Tmin, ea, Qg, Qg_cs):
    """
    Balanço de radiação de onda longa (BOL), segundo a equação de
    Stefan-Boltzmann corrigida pela FAO-56.

    Estima o saldo líquido de radiação de onda longa emitida pela
    superfície, corrigido pela nebulosidade (razão Qg/Qg_cs) e pela
    umidade do ar (via pressão de vapor atual, ea).

    Parâmetros
    ----------
    Tmax : float
        Temperatura máxima diária, em Kelvin.
    Tmin : float
        Temperatura mínima diária, em Kelvin.
    ea : float
        Pressão parcial (atual) de vapor d'água, em kPa.
    Qg : float
        Radiação solar global medida/estimada no dia, em MJ/m².
    Qg_cs : float
        Radiação solar de céu claro (clear-sky), em MJ/m².

    Retorna
    -------
    BOL : float
        Balanço de radiação de onda longa, em MJ/m² (valor negativo,
        representando perda líquida de energia por emissão terrestre).
    """
    termo_temp = 4.903e-9 * ((Tmax ** 4 + Tmin ** 4) / 2)
    termo_umidade = 0.34 - 0.14 * ea ** 0.5
    termo_nebulosidade = 1.35 * (Qg / Qg_cs) - 0.35
    BOL = -(termo_temp * termo_umidade * termo_nebulosidade)
    return BOL

## 1.9 — Capítulo 9: Evapo(transpi)ração

### Método de Thornthwaite (1948)

In [35]:
# Evapotranspiração potencial - Método de Thornthwaite (mensal)
def thornthwaite_mensal(df, col_T='T_media_C', lat=None):
    """
    Evapotranspiração potencial mensal (ETP) pelo método de Thornthwaite (1948).

    Estima a ETP a partir da temperatura média mensal do ar, corrigida pelo
    fotoperíodo (número de horas de brilho solar) em função da latitude e
    do número de dias de cada mês, conforme a formulação clássica do método.

    Parâmetros
    ----------
    df : pandas.DataFrame
        DataFrame com 12 linhas (Janeiro a Dezembro, nessa ordem) contendo
        ao menos a coluna de temperatura média mensal.
    col_T : str, opcional
        Nome da coluna de temperatura média mensal (°C) em `df`. Padrão 'T_media_C'.
    lat : float
        Latitude do local, em graus (negativa no hemisfério sul).

    Retorna
    -------
    df : pandas.DataFrame
        Cópia do DataFrame de entrada com a coluna adicional 'ETP_mm_mes'
        (evapotranspiração potencial corrigida, em mm/mês).
    """
    df = df.copy()
    T_mensal = df[col_T].to_numpy(dtype=float)
    dias_mes = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31])
    dia_juliano_medio = np.array([17, 47, 75, 105, 135, 162, 198, 228, 258, 288, 318, 344])

    i_mensal = np.where(T_mensal > 0, (T_mensal / 5) ** 1.514, 0)
    I = i_mensal.sum()
    a = 6.75e-7 * I ** 3 - 7.71e-5 * I ** 2 + 1.792e-2 * I + 0.49239
    ETP_nc = np.where(T_mensal > 0, 16 * (10 * T_mensal / I) ** a, 0)

    declinacao = 23.45 * sind(360 * (284 + dia_juliano_medio) / 365)
    tand_lat = sind(lat) / cosd(lat)
    tand_dec = sind(declinacao) / cosd(declinacao)
    ws = acosd(-tand_lat * tand_dec)
    N = (2 / 15) * ws
    K = (N / 12) * (dias_mes / 30)

    df['ETP_mm_mes'] = np.round(ETP_nc * K, 2)
    return df

### Aplicação 3
___
Um pesquisador do Oeste do Paraná deseja estimar a evapotranspiração potencial
(ETP) mensal de um município localizado em latitude 24,85° S, utilizando o
método de Thornthwaite (1948). Para isso, foram compiladas as temperaturas
médias mensais do ar (°C) referentes a um ano-padrão, conforme a tabela a
seguir:

| Mês | Jan | Fev | Mar | Abr | Mai | Jun | Jul | Ago | Set | Out | Nov | Dez |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| T média (°C) | 24,5 | 24,8 | 23,9 | 21,3 | 18,2 | 16,5 | 16,0 | 17,4 | 18,9 | 21,0 | 22,6 | 23,8 |

In [36]:
meses = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']
T_mensal = [24.5, 24.8, 23.9, 21.3, 18.2, 16.5, 16.0, 17.4, 18.9, 21.0, 22.6, 23.8]

df_in = pd.DataFrame({'Mes': meses, 'T_media_C': T_mensal})
df_out = thornthwaite_mensal(df_in, lat=-24.85)
df_out

,Mes,T_media_C,ETP_mm_mes
0,Jan,24.5,129.94
1,Fev,24.8,115.77
2,Mar,23.9,111.65
3,Abr,21.3,77.97
4,Mai,18.2,53.15
5,Jun,16.5,39.91
6,Jul,16.0,38.95
7,Ago,17.4,49.55
8,Set,18.9,61.64
9,Out,21.0,86.10


### Método de Camargo (modificado por Maluf)

In [37]:
# Evapotranspiração potencial - Método de Camargo (modificado por Maluf)
def camargo_maluf_mensal(df, col_T='T_media_C', lat=None):
    """
    Evapotranspiração potencial mensal (ETP) pelo método de Camargo (1971),
    com o coeficiente F modificado por Maluf (Camargo et al., 1999).

    Estima a ETP a partir da temperatura média mensal do ar e da radiação
    solar extraterrestre (Qo), calculada internamente a partir da declinação
    solar, do ângulo horário do nascer do Sol e da correção pela distância
    Terra-Sol, com coeficiente de ajuste F dependente da temperatura média
    anual do local. Função autocontida — não depende de funções auxiliares
    externas (sind, cosd, tand, irradiancia_extraterrestre, etc.).

    Parâmetros
    ----------
    df : pandas.DataFrame
        DataFrame com 12 linhas (Janeiro a Dezembro, nessa ordem) contendo
        ao menos a coluna de temperatura média mensal.
    col_T : str, opcional
        Nome da coluna de temperatura média mensal (°C) em `df`. Padrão 'T_media_C'.
    lat : float
        Latitude do local, em graus (negativa no hemisfério sul).

    Retorna
    -------
    df : pandas.DataFrame
        Cópia do DataFrame de entrada com as colunas adicionais 'Qo_MJ_m2dia'
        (radiação extraterrestre, MJ/m² dia), 'Qo_mm_dia' (equivalente de
        evaporação, mm/dia) e 'ETP_mm_mes' (ETP mensal, em mm/mês).
    """
    df = df.copy()
    T_mensal = df[col_T].to_numpy(dtype=float)
    dias_mes = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31])
    J = np.array([17, 47, 75, 105, 135, 162, 198, 228, 258, 288, 318, 344])  # dia juliano médio

    # Declinação solar (graus)
    declinacao = 23.45 * np.sin(np.radians(360 * (284 + J) / 365))

    # Ângulo horário do nascer do Sol (graus) - eq. 5.20
    tan_lat = np.sin(np.radians(lat)) / np.cos(np.radians(lat))
    tan_dec = np.sin(np.radians(declinacao)) / np.cos(np.radians(declinacao))
    Hn = np.degrees(np.arccos(np.clip(-tan_lat * tan_dec, -1, 1)))

    # Correção da distância Terra-Sol (d/D)^2 - eq. 5.30
    dD2 = 1 + 0.033 * np.cos(np.radians(360 * J / 365))

    # Irradiância solar extraterrestre diária (Qo), em MJ/m² dia
    hn_rad = np.radians(Hn)
    Qo_MJ = 37.6 * dD2 * (
        hn_rad * np.sin(np.radians(lat)) * np.sin(np.radians(declinacao))
        + np.cos(np.radians(lat)) * np.cos(np.radians(declinacao)) * np.sin(np.radians(Hn))
    )
    Qo_mm = 0.408 * Qo_MJ  # equivalente de evaporação, mm/dia

    # Coeficiente F em função da temperatura média anual (Camargo modif. Maluf)
    Tmed_anual = T_mensal.mean()
    if Tmed_anual < 23:
        F = 0.01
    elif Tmed_anual < 24:
        F = 0.0105
    else:
        F = 0.011

    ETP = F * Qo_mm * T_mensal * dias_mes

    df['Qo_MJ_m2dia'] = np.round(Qo_MJ, 2)
    df['Qo_mm_dia'] = np.round(Qo_mm, 2)
    df['ETP_mm_mes'] = np.round(ETP, 2)
    return df

### Aplicação 4
___

Utilizando os mesmos dados de temperatura média mensal do exemplo anterior
(latitude 24,85° S), estime a evapotranspiração potencial (ETP) mensal pelo
método de **Camargo (1971)**, com o coeficiente F ajustado conforme a
modificação de **Maluf** (Camargo et al., 1999).

In [38]:
meses = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']
T_mensal = [24.5, 24.8, 23.9, 21.3, 18.2, 16.5, 16.0, 17.4, 18.9, 21.0, 22.6, 23.8]

df_in = pd.DataFrame({'Mes': meses, 'T_media_C': T_mensal})
df_camargo = camargo_maluf_mensal(df_in, lat=-24.85)
df_camargo

,Mes,T_media_C,Qo_MJ_m2dia,Qo_mm_dia,ETP_mm_mes
0,Jan,24.5,42.53,17.35,131.81
1,Fev,24.8,39.89,16.28,113.02
2,Mar,23.9,35.46,14.47,107.20
3,Abr,21.3,29.47,12.02,76.83
4,Mai,18.2,24.15,9.85,55.59
5,Jun,16.5,21.58,8.80,43.58
6,Jul,16.0,22.61,9.23,45.76
7,Ago,17.4,26.97,11.00,59.34
8,Set,18.9,32.85,13.40,75.98
9,Out,21.0,38.19,15.58,101.43


### Método de Hargreaves & Samani (1985)

In [39]:
# ETP por Hargreaves & Samani (1985)
def etp_hargreaves_samani(Qo, Tmax, Tmin, Tmed):
    """
    Evapotranspiração potencial pelo método de
    Hargreaves & Samani (1985), que usa a amplitude térmica diária como
    substituto indireto da nebulosidade/radiação solar efetiva.

    Parâmetros
    ----------
    Qo : float
        Irradiância solar extraterrestre, em MJ/m² dia.
    Tmax : float
        Temperatura máxima do ar, em °C.
    Tmin : float
        Temperatura mínima do ar, em °C.
    Tmed : float
        Temperatura média do ar, em °C.

    Retorna
    -------
    ETP : float
        Evapotranspiração potencial, em mm/dia.
    """
    Qo_mm = 0.408 * Qo  # equivalente de evaporação, mm/dia
    return 0.0023 * Qo_mm * (Tmax - Tmin) ** 0.5 * (Tmed + 17.8)


### Aplicação 5
___
Determine a Evapotranspiração potencial pelo método de Hargreaves-Samani. Considere a cidade de Foz do Iguaçu-PR (Latitude = -25,60°, atitude = 235,09 m) no dia 21/05.
OBS. Tmax = 21,2 ºC. Tmin = 7,4 ºC; URmax =  97,1%; Qo: 22.84 MJ/m²d; e, URmin  = 66,8%

In [40]:
# Dados
Tmax = 21.2
Tmin = 7.4
URmax = 0.971
URmin = 0.668
lat = -25.6
dia = 21
mes = 5
dia = 21
mes = 5
Qo = 22.8

Tmed = (Tmax + Tmin) / 2

ETP = etp_hargreaves_samani(Qo, Tmax, Tmin, Tmed)
print(f"Evapotranspiração potencial (ETP_HS): {ETP:.2f} mm/dia")


Evapotranspiração potencial (ETP_HS): 2.55 mm/dia


### Método de Priestley & Taylor (1972)

In [41]:
# Declive da curva de pressão de saturação de vapor - Delta
def declive_pressao_vapor(T_ar):
    """
    Declive da curva de pressão de saturação de vapor (Delta), na
    temperatura do ar considerada.

    Obtida derivando-se a equação de Tetens (eq. 7.2) em relação à
    temperatura. Necessária nos métodos combinados (energia + aerodinâmico),
    como Priestley-Taylor e Penman-Monteith.

    Parâmetros
    ----------
    T_ar : float
        Temperatura do ar, em °C (em geral, a temperatura média diária).

    Retorna
    -------
    Delta : float
        Declive da curva de pressão de saturação de vapor, em kPa/°C.
    """
    es = es_tetens(T_ar)
    return 4098 * es / (T_ar + 237.3) ** 2

In [42]:
# ETP por Priestley-Taylor (1972)
def etp_priestley_taylor(Rn, G, Delta, gamma, alfa=1.26):
    """
    Evapotranspiração potencial pelo método de Priestley & Taylor (1972).

    Simplificação do termo aerodinâmico da equação de Penman, válida para
    superfícies bem supridas de água. O coeficiente empírico alfa (1,26)
    compensa a advecção não contabilizada quando se ignora o termo
    aerodinâmico.

    Parâmetros
    ----------
    Rn : float
        Saldo de radiação, em MJ/m² dia.
    G : float
        Fluxo de calor no solo, em MJ/m² dia (geralmente desprezado,
        G = 0, em escala diária).
    Delta : float
        Declive da curva de pressão de saturação de vapor, em kPa/°C.
    gamma : float
        Constante psicrométrica, em kPa/°C.
    alfa : float, opcional
        Coeficiente de Priestley-Taylor (padrão 1,26).

    Retorna
    -------
    ETP : float
        Evapotranspiração potencial, em mm/dia.
    """
    lambda_v = 2.45  # calor latente de vaporização, MJ/kg
    return alfa * (Delta / (Delta + gamma)) * (Rn - G) / lambda_v

### Aplicação 6
___
Determine a evapotranspiração potencial pelo método de Priestley-Taylor (1972) para a cidade de Foz do Iguaçu-PR (Latitude = -25,60°, altitude = 235,09 m), no dia 21/05. Considere Tmax = 21,2 ºC, Tmin = 7,4 ºC, URmax = 97,1% e URmin = 66,8%. Utilize Qg = Qg_HS = 13,58 MJ/m² dia (Aplicação 2/5), albedo r = 0,23 (referência FAO-56) e G = 0.

In [43]:
# Dados
Tmax = 21.2
Tmin = 7.4
URmax = 0.971
URmin = 0.668
lat = -25.6
alt = 235.09
dia = 21
mes = 5
G = 0

# Resolução
NDA = nda(dia, mes)
delta = declinacao_solar(NDA)
Hn = angulo_horario_nascer(lat, delta)
dD2 = fator_correcao_distancia(NDA)
Qo = irradiancia_extraterrestre(lat, delta, Hn, dD2)
Qg = Qg_hargreaves(Tmax, Tmin, Qo)
print(f"Radiação solar global (Qg): {Qg:.2f} MJ/m² dia")

Tmed = (Tmax + Tmin) / 2
es_max = es_tetens(Tmax)
es_min = es_tetens(Tmin)
es = (es_max + es_min) / 2
ea = (es_min * URmax + es_max * URmin) / 2
print(f"Pressão de saturação (es): {es:.3f} kPa | Pressão atual (ea): {ea:.3f} kPa")

Patm = patm_altitude(alt)
gamma = constante_psicrometrica(Patm)
Delta = declive_pressao_vapor(Tmed)
print(f"gamma: {gamma:.4f} kPa/°C | Delta: {Delta:.4f} kPa/°C")

Qg_cs = (0.75 + 2e-5 * alt) * Qo  # radiação de céu claro (FAO-56)
BOC = boc_saldo(Qg, r=0.23)
BOL = bol_saldo(Tmax, Tmin, ea, Qg, Qg_cs)
Rn = saldo_radiacao(BOC, BOL)
print(f"BOC: {BOC:.2f} | BOL: {BOL:.6f} | Rn: {Rn:.2f} MJ/m² dia")

ETP_PT = etp_priestley_taylor(Rn, G, Delta, gamma)
print(f"Evapotranspiração potencial (ETP_PT): {ETP_PT:.2f} mm/dia")

Radiação solar global (Qg): 13.58 MJ/m² dia
Pressão de saturação (es): 1.774 kPa | Pressão atual (ea): 1.341 kPa
gamma: 0.0655 kPa/°C | Delta: 0.1055 kPa/°C
BOC: 10.45 | BOL: -0.000064 | Rn: 10.45 MJ/m² dia
Evapotranspiração potencial (ETP_PT): 3.32 mm/dia


### Método de Penman-Monteith — padrão FAO (Allen et al., 1998)

In [44]:
# ETo por Penman-Monteith - padrão FAO-56 (Allen et al., 1998)
def eto_penman_monteith_fao56(Rn, G, Tmed, u2, es, ea, Delta, gamma):
    """
    Evapotranspiração de referência (ETo) pela equação de
    Penman-Monteith, padronizada pelo boletim FAO-56 (Allen et al., 1998).

    Combina os termos de energia (radiativo) e aerodinâmico (advectivo),
    referenciados a uma cultura hipotética (grama, altura 0,12 m, albedo
    0,23, resistência de superfície fixa), sendo o método-padrão
    internacional para estimativa da evapotranspiração de referência.

    Parâmetros
    ----------
    Rn : float
        Saldo de radiação, em MJ/m² dia.
    G : float
        Fluxo de calor no solo, em MJ/m² dia (G = 0 para escala diária).
    Tmed : float
        Temperatura média diária do ar, em °C.
    u2 : float
        Velocidade do vento a 2 m de altura, em m/s.
    es : float
        Pressão de saturação de vapor, em kPa.
    ea : float
        Pressão parcial (atual) de vapor d'água, em kPa.
    Delta : float
        Declive da curva de pressão de saturação de vapor, em kPa/°C.
    gamma : float
        Constante psicrométrica, em kPa/°C.

    Retorna
    -------
    ETo : float
        Evapotranspiração de referência, em mm/dia.
    """
    numerador = 0.408 * Delta * (Rn - G) + gamma * (900 / (Tmed + 273)) * u2 * (es - ea)
    denominador = Delta + gamma * (1 + 0.34 * u2)
    return numerador / denominador

### Aplicação 7
___
Determine a evapotranspiração de referência (ETo) pelo método de Penman-Monteith FAO-56 para a cidade de Foz do Iguaçu-PR (Latitude = -25,60°, altitude = 235,09 m), no dia 21/05. Considere os mesmos dados da Aplicação 6 (Tmax = 21,2 ºC, Tmin = 7,4 ºC, URmax = 97,1%, URmin = 66,8%) e velocidade do vento a 2 m de altura u2 = 2,0 m/s.

In [45]:
# Dados
Tmax = 21.2
Tmin = 7.4
URmax = 0.971
URmin = 0.668
lat = -25.6
alt = 235.09
dia = 21
mes = 5
G = 0
u2 = 2.0  # velocidade do vento a 2 m, em m/s

# Resolução (reaproveitando Tmed, es, ea, Delta, gamma, Rn, G da Aplicação 6)
ET0_PM = eto_penman_monteith_fao56(Rn, G, Tmed, u2, es, ea, Delta, gamma)
print(f"Evapotranspiração de referência (ETo_PM): {ET0_PM:.2f} mm/dia")

Evapotranspiração de referência (ETo_PM): 2.91 mm/dia


## 2 — Capítulo 10: Grau-dias

In [46]:

def data_maturacao_fisiologica(df, Tb, CT, dia_semeadura, mes_semeadura, intervalo='d', ano=2023):
    """
    Calcula a data de maturação fisiológica de uma cultura, a partir da
    data de semeadura, por acúmulo de graus-dia (GDA) até atingir a
    constante térmica do ciclo (CT).

    Regra de cálculo do GD diário (GDi):
    - Se Tb < Tmin:  GDi = Tmed - Tb
    - Se Tb >= Tmin: GDi = (Tmax - Tb)^2 / [2*(Tmax - Tmin)]

    Parâmetros
    ----------
    df : pandas.DataFrame
        Colunas: 'dia', 'mes', 'Tmed', 'Tmax', 'Tmin' (uma linha por período,
        em ordem cronológica). No mensal, 'dia' é só um marcador (ex.: 1);
        no decendial, 'dia' é o dia de início do decêndio (1, 11 ou 21); no
        diário, 'dia' é o dia real do mês.
    Tb : float
        Temperatura base da cultura, em °C.
    CT : float
        Soma térmica total do ciclo (constante térmica), em °C·dia.
    dia_semeadura, mes_semeadura : int
        Dia e mês da semeadura.
    intervalo : str, opcional
        'd' (diário), 'M' (mensal) ou 'dec' (decendial, sempre 10 dias). Padrão 'd'.
    ano : int, opcional
        Ano de referência (padrão 2023, não-bissexto).

    Retorna
    -------
    resultado : pandas.DataFrame
        Colunas 'data' e 'GD_ciclo' (GDA acumulado), da semeadura até a maturação.
    """
    MESES_PT = {1:'janeiro',2:'fevereiro',3:'março',4:'abril',5:'maio',6:'junho',
            7:'julho',8:'agosto',9:'setembro',10:'outubro',11:'novembro',12:'dezembro'}

    df = df.reset_index(drop=True)

    if intervalo == 'M':
        idx_ref = df.index[df['mes'] == mes_semeadura]
    else:
        idx_ref = df.index[(df['mes'] == mes_semeadura) & (df['dia'] == dia_semeadura)]
    if len(idx_ref) == 0:
        raise ValueError(f"Não encontrei a linha com dia={dia_semeadura}, mes={mes_semeadura} no df.")
    idx_ref = idx_ref[0]

    n_linhas = len(df)
    ordem = [(idx_ref + i) % n_linhas for i in range(n_linhas)]

    registros = []
    acumulado = 0.0
    data_final = None

    for pos, i in enumerate(ordem):
        row = df.loc[i]
        Tmed, Tmax, Tmin = row['Tmed'], row['Tmax'], row['Tmin']
        mes_row = int(row['mes'])

        if Tb < Tmin:
            GDi = Tmed - Tb
        else:
            GDi = (Tmax - Tb) ** 2 / (2 * (Tmax - Tmin))

        if intervalo == 'd':
            n_periodo = 1
        elif intervalo == 'dec':
            n_periodo = 10
        elif intervalo == 'M':
            n_periodo = monthrange(ano, mes_row)[1]
        else:
            raise ValueError("intervalo deve ser 'd', 'M' ou 'dec'")

        if pos == 0 and intervalo == 'M':
            n_efetivo = n_periodo - dia_semeadura
        else:
            n_efetivo = n_periodo

        GD_periodo = GDi * n_efetivo
        acumulado_anterior = acumulado
        acumulado += GD_periodo

        if intervalo in ('d', 'dec'):
            data_periodo = date(ano, mes_row, int(row['dia']))
        else:
            data_periodo = date(ano, mes_row, dia_semeadura if pos == 0 else 1)

        registros.append({'data': data_periodo, 'GD_ciclo': round(acumulado, 2)})

        if acumulado >= CT:
            if intervalo == 'd':
                data_final = data_periodo
            else:
                faltante = CT - acumulado_anterior
                dias_necessarios = int(np.ceil(faltante / GDi))
                dias_necessarios = min(dias_necessarios, n_efetivo)
                data_final = data_periodo + timedelta(days=dias_necessarios - 1)
            break

    resultado = pd.DataFrame(registros)
    print(f"Data de semeadura: {dia_semeadura:02d} de {MESES_PT[mes_semeadura]}")
    print(f"Data de maturação fisiológica: {data_final.day:02d} de {MESES_PT[data_final.month]}")
    return resultado


def data_semeadura(df, Tb, CT, dia_maturacao, mes_maturacao, intervalo='d', ano=2023):
    """
    Calcula a data de semeadura necessária para que uma cultura atinja a
    maturação fisiológica em uma data de referência conhecida (ex.: data
    de colheita desejada), por acúmulo retroativo de graus-dia (GDA) até
    a constante térmica do ciclo (CT).

    Regra de cálculo do GD diário (GDi):
    - Se Tb < Tmin:  GDi = Tmed - Tb
    - Se Tb >= Tmin: GDi = (Tmax - Tb)^2 / [2*(Tmax - Tmin)]

    Parâmetros
    ----------
    df : pandas.DataFrame
        Colunas: 'dia', 'mes', 'Tmed', 'Tmax', 'Tmin' (uma linha por período,
        em ordem cronológica). No mensal, 'dia' é só um marcador (ex.: 1);
        no decendial, 'dia' é o dia de início do decêndio (1, 11 ou 21); no
        diário, 'dia' é o dia real do mês.
    Tb : float
        Temperatura base da cultura, em °C.
    CT : float
        Soma térmica total do ciclo, em °C·dia.
    dia_maturacao, mes_maturacao : int
        Dia e mês da maturação (data de referência conhecida).
    intervalo : str, opcional
        'd' (diário), 'M' (mensal) ou 'dec' (decendial, sempre 10 dias). Padrão 'd'.
    ano : int, opcional
        Ano de referência (padrão 2023, não-bissexto).

    Retorna
    -------
    resultado : pandas.DataFrame
        Colunas 'data' e 'GD_ciclo' (GDA acumulado), da maturação (referência)
        até a semeadura.
    """
    MESES_PT = {1:'janeiro',2:'fevereiro',3:'março',4:'abril',5:'maio',6:'junho',
            7:'julho',8:'agosto',9:'setembro',10:'outubro',11:'novembro',12:'dezembro'}

    df = df.reset_index(drop=True)

    if intervalo == 'M':
        idx_ref = df.index[df['mes'] == mes_maturacao]
    else:
        idx_ref = df.index[(df['mes'] == mes_maturacao) & (df['dia'] == dia_maturacao)]
    if len(idx_ref) == 0:
        raise ValueError(f"Não encontrei a linha com dia={dia_maturacao}, mes={mes_maturacao} no df.")
    idx_ref = idx_ref[0]

    n_linhas = len(df)
    ordem = [(idx_ref - i) % n_linhas for i in range(n_linhas)]

    registros = []
    acumulado = 0.0
    data_sem = None

    for pos, i in enumerate(ordem):
        row = df.loc[i]
        Tmed, Tmax, Tmin = row['Tmed'], row['Tmax'], row['Tmin']
        mes_row = int(row['mes'])

        if Tb < Tmin:
            GDi = Tmed - Tb
        else:
            GDi = (Tmax - Tb) ** 2 / (2 * (Tmax - Tmin))

        if intervalo == 'd':
            n_periodo = 1
        elif intervalo == 'dec':
            n_periodo = 10
        elif intervalo == 'M':
            n_periodo = monthrange(ano, mes_row)[1]
        else:
            raise ValueError("intervalo deve ser 'd', 'M' ou 'dec'")

        if pos == 0 and intervalo == 'M':
            n_efetivo = dia_maturacao
        else:
            n_efetivo = n_periodo

        GD_periodo = GDi * n_efetivo
        acumulado_anterior = acumulado
        acumulado += GD_periodo

        if intervalo in ('d', 'dec'):
            data_periodo = date(ano, mes_row, int(row['dia']))
        else:
            data_periodo = date(ano, mes_row, 1)

        registros.append({'data': data_periodo, 'GD_ciclo': round(acumulado, 2)})

        if acumulado >= CT:
            if intervalo == 'd':
                data_sem = data_periodo
            else:
                faltante = CT - acumulado_anterior
                dias_necessarios = int(np.ceil(faltante / GDi))
                dias_necessarios = min(dias_necessarios, n_efetivo)
                data_sem = data_periodo + timedelta(days=dias_necessarios - 1)
            break

    resultado = pd.DataFrame(registros)
    print(f"Data de maturação (referência): {dia_maturacao:02d} de {MESES_PT[mes_maturacao]}")
    print(f"Data de semeadura necessária: {data_sem.day:02d} de {MESES_PT[data_sem.month]}")
    return resultado

### Aplicação 8
___
Para a cultura da soja, semeada em Londrina-PR no dia 12/11, deseja-se estimar a data de maturação fisiológica por acúmulo de graus-dia. Considere temperatura base Tb = 14 °C e constante térmica do ciclo CT = 1030 °C·dia. Utilize as normais climatológicas mensais de Londrina (1991–2020) e intervalo de dados mensal.

In [47]:
# Dados de entrada
meses = list(range(1, 13))
Tmed  = [24.7, 24.7, 23.4, 22.5, 18.9, 17.6, 17.6, 19.6, 21.4, 23.1, 23.9, 24.8]
Tmin  = [20.4, 20.4, 18.6, 17.4, 13.9, 12.7, 12.0, 13.4, 15.5, 17.6, 18.5, 19.9]
Tmax  = [29.7, 30.0, 29.3, 28.0, 24.9, 23.5, 23.8, 25.9, 26.9, 28.0, 28.9, 29.6]  # ilustrativo

Tb = 14
CT = 1030
dia_semeadura = 12
mes_semeadura = 11  # novembro

df = pd.DataFrame({
    'dia': [1]*dia_semeadura,
    'mes': meses,
    'Tmed': Tmed,
    'Tmax': Tmax,
    'Tmin': Tmin
})
df

,dia,mes,Tmed,Tmax,Tmin
0,1,1,24.7,29.7,20.4
1,1,2,24.7,30.0,20.4
2,1,3,23.4,29.3,18.6
3,1,4,22.5,28.0,17.4
4,1,5,18.9,24.9,13.9
5,1,6,17.6,23.5,12.7
6,1,7,17.6,23.8,12.0
7,1,8,19.6,25.9,13.4
8,1,9,21.4,26.9,15.5
9,1,10,23.1,28.0,17.6


In [48]:
# Resolução
resultado = data_maturacao_fisiologica(df, Tb, CT, dia_semeadura, mes_semeadura, intervalo='M')
resultado

Data de semeadura: 12 de novembro
Data de maturação fisiológica: 18 de fevereiro


,data,GD_ciclo
0,2023-11-12,178.2
1,2023-12-01,513.0
2,2023-01-01,844.7
3,2023-02-01,1144.3


### Aplicação 9
___
Um produtor de Londrina-PR deseja que a cultura Y atinja a maturação fisiológica até o dia 22/06, e precisa saber até quando pode semear. Considere temperatura base Tb = 10 °C e constante térmica do ciclo CT = 800 °C·dia. Utilize as normais climatológicas mensais de Londrina (1991–2020) e intervalo de dados mensal.

In [49]:
# Dados de entrada
Tb = 10
CT = 800
dia_maturacao = 22
mes_maturacao = 6  # junho

meses = list(range(1, 13))
Tmed  = [24.7, 24.7, 23.4, 22.5, 18.9, 17.6, 17.6, 19.6, 21.4, 23.1, 23.9, 24.8]
Tmin  = [20.4, 20.4, 18.6, 17.4, 13.9, 12.7, 12.0, 13.4, 15.5, 17.6, 18.5, 19.9]
Tmax  = [29.7, 30.0, 29.3, 28.0, 24.9, 23.5, 23.8, 25.9, 26.9, 28.0, 28.9, 29.6]  # ilustrativo

df = pd.DataFrame({
    'dia': [1]*12,
    'mes': meses,
    'Tmed': Tmed,
    'Tmax': Tmax,
    'Tmin': Tmin
})
df

,dia,mes,Tmed,Tmax,Tmin
0,1,1,24.7,29.7,20.4
1,1,2,24.7,30.0,20.4
2,1,3,23.4,29.3,18.6
3,1,4,22.5,28.0,17.4
4,1,5,18.9,24.9,13.9
5,1,6,17.6,23.5,12.7
6,1,7,17.6,23.8,12.0
7,1,8,19.6,25.9,13.4
8,1,9,21.4,26.9,15.5
9,1,10,23.1,28.0,17.6


In [50]:
# Cálculo
resultado = data_semeadura(df, Tb, CT, dia_maturacao, mes_maturacao, intervalo='M')
resultado

Data de maturação (referência): 22 de junho
Data de semeadura necessária: 29 de abril


,data,GD_ciclo
0,2023-06-01,167.2
1,2023-05-01,443.1
2,2023-04-01,818.1


## 3 — Capítulo 11: Balanço Hídrico Climatológico

In [51]:
def balanco_hidrico_climatologico(df, CAD=100.0):
    """
    Calcula o Balanço Hídrico Climatológico (BHC), pelo método de
    Thornthwaite & Mather (1955), a partir de uma série de precipitação
    (P) e evapotranspiração potencial (ETP).

    Parâmetros
    ----------
    df : pandas.DataFrame
        Colunas obrigatórias: 'Meses', 'P (mm/mês)', 'ETP (mm/mês)'.
    CAD : float, opcional
        Capacidade de água disponível no solo, em mm. Padrão 100.0.

    Retorna
    -------
    df_bh : pandas.DataFrame
        Cópia do df de entrada, acrescido das colunas:
        'CAD', 'P-ETP', 'ARM (mm/mês)', 'NEG.ACUM (mm)', 'ALT (mm/mês)',
        'ETR (mm/mês)', 'DEF (mm/mês)', 'EXC (mm/mês)'.
    """
    df_bh = df.copy()

    # ============================================================
    # 1) Parâmetros
    # ============================================================
    df_bh['CAD'] = CAD

    # ============================================================
    # Saldo hídrico
    # ============================================================
    df_bh['P-ETP'] = df_bh['P (mm/mês)'] - df_bh['ETP (mm/mês)']

    # ============================================================
    # Inicialização
    # ============================================================
    ARM = [df_bh['CAD'].iloc[0]]      # solo cheio no início
    NEG_ACUM = [0.0]                  # sem déficit acumulado inicial

    # ============================================================
    # ARM – Armazenamento de água no solo (loop mensal)
    # ============================================================
    for i in range(len(df_bh)):
        p_etp = df_bh['P-ETP'].iloc[i]
        cad = df_bh['CAD'].iloc[i]
        arm_prev = ARM[-1]
        neg_prev = NEG_ACUM[-1]

        if p_etp < 0:
            # Acumula déficit
            neg = neg_prev + p_etp
            arm = cad * np.exp(neg / cad)
        else:
            # Reposição hídrica
            arm = min(arm_prev + p_etp, cad)

            # Recalcula NEG.ACUM pela inversão
            if arm < cad:
                neg = cad * np.log(arm / cad)
            else:
                neg = 0.0

        ARM.append(arm)
        NEG_ACUM.append(neg)

    # Remove o valor inicial extra
    df_bh['ARM (mm/mês)'] = ARM[1:]
    df_bh['NEG.ACUM (mm)'] = NEG_ACUM[1:]

    # ============================================================
    # ALT – Variação do armazenamento
    # ============================================================
    df_bh['ALT (mm/mês)'] = df_bh['ARM (mm/mês)'].diff().fillna(0)

    # ============================================================
    # ETR – Evapotranspiração real
    # ============================================================
    df_bh['ETR (mm/mês)'] = np.where(
        df_bh['P-ETP'] < 0,
        df_bh['P (mm/mês)'] + df_bh['ALT (mm/mês)'].abs(),
        df_bh['ETP (mm/mês)']
    )

    # ============================================================
    # DEF – Deficiência hídrica
    # ============================================================
    df_bh['DEF (mm/mês)'] = df_bh['ETP (mm/mês)'] - df_bh['ETR (mm/mês)']

    # ============================================================
    # EXC – Excedente hídrico
    # ============================================================
    df_bh['EXC (mm/mês)'] = np.where(
        (df_bh['P-ETP'] > 0) & (df_bh['ARM (mm/mês)'] == df_bh['CAD']),
        df_bh['P-ETP'] - df_bh['ALT (mm/mês)'],
        0
    )

    return df_bh

### Aplicação 10
___
Com base nos dados de ETP e P, determine a BHC, considerando a CAD = 100 mm

In [52]:
dados_bhc = {
    'Meses': ['jan', 'fev', 'mar', 'abr', 'maio', 'jun', 'jul', 'ago', 'set', 'out', 'nov', 'dez'],
    'P (mm/mês)': [203.400000, 189.533333, 196.266667, 116.933333, 14.033333, 3.333333,
                   1.300000, 4.366667, 18.033333, 99.466667, 223.733333, 234.966667],
    'ETP (mm/mês)': [128.730000, 106.286667, 117.300000, 109.263333, 106.800000, 102.033333,
                      114.986667, 134.523333, 141.426667, 149.146667, 118.616667, 119.960000]
}

In [53]:
df_ini = pd.DataFrame(dados_bhc)
df_bh = balanco_hidrico_climatologico(df_ini, CAD=100.0)
df_bh

,Meses,P (mm/mês),ETP (mm/mês),CAD,P-ETP,ARM (mm/mês),NEG.ACUM (mm),ALT (mm/mês),ETR (mm/mês),DEF (mm/mês),EXC (mm/mês)
0,jan,203.400000,128.730000,100.0,74.670000,100.000000,0.000000,0.000000,128.730000,0.000000,74.670000
1,fev,189.533333,106.286667,100.0,83.246666,100.000000,0.000000,0.000000,106.286667,0.000000,83.246666
2,mar,196.266667,117.300000,100.0,78.966667,100.000000,0.000000,0.000000,117.300000,0.000000,78.966667
3,abr,116.933333,109.263333,100.0,7.670000,100.000000,0.000000,0.000000,109.263333,0.000000,7.670000
4,maio,14.033333,106.800000,100.0,-92.766667,39.547541,-92.766667,-60.452459,74.485792,32.314208,0.000000
5,jun,3.333333,102.033333,100.0,-98.700000,14.739095,-191.466667,-24.808446,28.141779,73.891554,0.000000
6,jul,1.300000,114.986667,100.0,-113.686667,4.728636,-305.153334,-10.010459,11.310459,103.676208,0.000000
7,ago,4.366667,134.523333,100.0,-130.156666,1.286686,-435.310000,-3.441950,7.808617,126.714716,0.000000
8,set,18.033333,141.426667,100.0,-123.393334,0.374612,-558.703334,-0.912074,18.945407,122.481260,0.000000
9,out,99.466667,149.146667,100.0,-49.680000,0.227942,-608.383334,-0.146670,99.613337,49.533330,0.000000


## 4 — Capítulo 11: Balanço Hídrico de Cultura

In [54]:

def balanco_hidrico_cultura(df):
    """
    Calcula o Balanço Hídrico de Cultura (BHc), pelo método de
    Thornthwaite & Mather, a partir de um df já estruturado com Chuva,
    ETc e CAD por período.

    Independente da escala temporal (diária, decendial, mensal etc.) —
    o usuário é responsável por pré-calcular 'ETc' (= Kc x ETo, já
    considerando a fase fenológica da cultura) e 'CAD' (= z x DTA, já
    considerando o avanço da profundidade radicular) na escala desejada;
    esta função só executa a contabilidade hídrica período a período.

    Parâmetros
    ----------
    df : pandas.DataFrame
        Colunas obrigatórias: 'Chuva' (mm/período), 'ETc' (mm/período) e
        'CAD' (mm, capacidade de água disponível no período), em ordem
        cronológica.

    Retorna
    -------
    df_bhc : pandas.DataFrame
        Cópia do df de entrada, acrescido de: 'P-ETc', 'ARM', 'ALT',
        'ETR', 'DEF', 'EXC', 'ISNA'.
    """
    df_bhc = df.copy()

    # Saldo hídrico
    df_bhc['P-ETc'] = df_bhc['Chuva'] - df_bhc['ETc']

    # ARM (armazenamento de água no solo)
    PETc = df_bhc['P-ETc'].to_numpy()
    CAD = df_bhc['CAD'].to_numpy()

    ARM = [CAD[0]]  # solo cheio na CAD do primeiro período
    for p, cad in zip(PETc, CAD):
        prev = ARM[-1]
        if p < 0:
            ARM.append(prev * np.exp(p / cad))
        elif p + prev >= cad:
            ARM.append(cad)
        else:
            ARM.append(prev + p)
    ARM = ARM[1:]
    df_bhc['ARM'] = ARM

    # ALT (variação de ARM entre períodos)
    ALT = [0] + list(np.array(ARM[1:]) - np.array(ARM[:-1]))
    df_bhc['ALT'] = ALT

    # ETR, DEF, EXC, ISNA
    df_bhc['ETR'] = np.where(
        df_bhc['P-ETc'] < 0,
        df_bhc['Chuva'] + df_bhc['ALT'].abs(),
        df_bhc['ETc']
    )
    df_bhc['DEF'] = df_bhc['ETc'] - df_bhc['ETR']
    df_bhc['EXC'] = np.where(
        df_bhc['ARM'] < df_bhc['CAD'],
        0,
        df_bhc['P-ETc'] - df_bhc['ALT']
    )
    df_bhc['ISNA'] = df_bhc['ETR'] / df_bhc['ETc']

    return df_bhc

### Aplicação 11
___
Com base nos dados de ETc, P e CAD, determine a BH de cultura

In [55]:
dados_bhc_cultura = {
    'Chuva': [45.2, 38.7, 22.4, 15.1, 8.3, 5.6, 4.2, 9.8, 18.5, 35.9, 48.3, 52.1],
    'ETc':   [17.6, 23.93, 33.9, 44.46, 54.16, 53.47, 52.78, 49.28, 41.04, 34.0, 28.42, 24.54],
    'CAD':   [22.5, 37.5, 52.5, 67.5, 82.5, 90.0, 90.0, 90.0, 90.0, 90.0, 90.0, 90.0]
}

df_ini = pd.DataFrame(dados_bhc_cultura)
df_ini

,Chuva,ETc,CAD
0,45.2,17.60,22.5
1,38.7,23.93,37.5
2,22.4,33.90,52.5
3,15.1,44.46,67.5
4,8.3,54.16,82.5
5,5.6,53.47,90.0
6,4.2,52.78,90.0
7,9.8,49.28,90.0
8,18.5,41.04,90.0
9,35.9,34.00,90.0


In [56]:
df_bhc = balanco_hidrico_cultura(df_ini)
df_bhc

,Chuva,ETc,CAD,P-ETc,ARM,ALT,ETR,DEF,EXC,ISNA
0,45.2,17.60,22.5,27.60,22.500000,0.000000,17.600000,0.000000,27.6,1.000000
1,38.7,23.93,37.5,14.77,37.270000,14.770000,23.930000,0.000000,0.0,1.000000
2,22.4,33.90,52.5,-11.50,29.938375,-7.331625,29.731625,4.168375,0.0,0.877039
3,15.1,44.46,67.5,-29.36,19.378770,-10.559605,25.659605,18.800395,0.0,0.577139
4,8.3,54.16,82.5,-45.86,11.115042,-8.263728,16.563728,37.596272,0.0,0.305830
5,5.6,53.47,90.0,-47.87,6.530023,-4.585019,10.185019,43.284981,0.0,0.190481
6,4.2,52.78,90.0,-48.58,3.806205,-2.723818,6.923818,45.856182,0.0,0.131183
7,9.8,49.28,90.0,-39.48,2.454605,-1.351600,11.151600,38.128400,0.0,0.226291
8,18.5,41.04,90.0,-22.54,1.910799,-0.543806,19.043806,21.996194,0.0,0.464030
9,35.9,34.00,90.0,1.90,3.810799,1.900000,34.000000,0.000000,0.0,1.000000
